# 06 — FNO architecture

A spectral layer applies a learned transformation to selected Fourier coefficients:

\[
v_{k+1}(x)=\sigma\left(Wv_k(x)+\mathcal{F}^{-1}\left(R_\theta\,\mathcal{F}(v_k)\right)(x)\right).
\]

The implementation in `src/oisst_fno/model.py` is intentionally compact so its mechanics remain inspectable.

In [ ]:
import torch

from oisst_fno.metrics import parameter_count
from oisst_fno.model import FNO2d, TruncatedFourierMix2d

layer = TruncatedFourierMix2d(in_channels=4, out_channels=6, modes_y=8, modes_x=8)
x = torch.randn(2, 4, 48, 64)
y = layer(x)
print("spectral layer:", x.shape, "->", y.shape)

In [ ]:
LOOKBACK = 14
# 14 SST channels + 1 ocean-mask channel. Coordinates are appended internally.
model = FNO2d(
    in_channels=LOOKBACK + 1,
    out_channels=1,
    width=48,
    modes_y=16,
    modes_x=16,
    depth=4,
    padding=8,
)
print(model)
print(f"trainable parameters: {parameter_count(model):,}")

In [ ]:
probe = torch.randn(2, LOOKBACK + 1, 81, 101)
out = model(probe)
print(probe.shape, "->", out.shape)
assert out.shape == (2, 1, 81, 101)

### What the architecture does *not* guarantee

- Fourier layers do not make the model physically correct.
- Retaining low modes can smooth high-frequency structure.
- Coordinate channels and padding matter because the regional domain is not periodic.
- “Resolution invariant” is a theoretical/operator-level concept that must be tested empirically for this implementation.